# 🥉 Template: Bronze Layer - Ingestão de Dados

## 📋 Objetivo
Este template demonstra como carregar dados brutos na camada Bronze seguindo as melhores práticas do Microsoft Fabric.

## 🔧 Configurações
Adapte as variáveis abaixo para seu projeto específico:

```python
# TODO: Configurar variáveis do projeto
WORKSPACE_NAME = "SEU_WORKSPACE"  # Nome do workspace Fabric
LAKEHOUSE_NAME = "SEU_LAKEHOUSE"  # Nome do lakehouse
DATA_SOURCE = "SUA_FONTE"         # Fonte dos dados (csv, parquet, api, etc.)
```

In [ ]:
# 📦 Importações e Configurações Iniciais
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import logging
from datetime import datetime

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Inicializar Spark session
spark = SparkSession.builder.appName("BronzeIngestion").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("✅ Configurações iniciais completas")

In [ ]:
# 🎯 Configurações do Projeto
# TODO: Personalize estas configurações para seu projeto

# Configurações do Lakehouse
LAKEHOUSE_PATH = "/lakehouse/default/"
BRONZE_PATH = f"{LAKEHOUSE_PATH}Tables/bronze"

# Configurações de origem dos dados
SOURCE_PATH = "/lakehouse/default/Files/raw_data/"  # TODO: Ajustar caminho
SOURCE_FORMAT = "csv"  # TODO: Ajustar formato (csv, parquet, json, etc.)

# Tabelas para processar
# TODO: Definir tabelas específicas do seu domínio
TABLES_CONFIG = {
    "customers": {
        "source_file": "customers.csv",
        "partition_columns": ["processing_date"],
        "schema": None  # Auto-inferir ou definir schema específico
    },
    "orders": {
        "source_file": "orders.csv",
        "partition_columns": ["processing_date"],
        "schema": None
    },
    "products": {
        "source_file": "products.csv",
        "partition_columns": ["processing_date"],
        "schema": None
    }
}

# Timestamp de processamento
PROCESSING_TIMESTAMP = datetime.now()
PROCESSING_DATE = PROCESSING_TIMESTAMP.strftime("%Y-%m-%d")

print(f"📅 Data de processamento: {PROCESSING_DATE}")
print(f"🎯 Caminho Bronze: {BRONZE_PATH}")
print(f"📂 Origem dos dados: {SOURCE_PATH}")

In [ ]:
# 🔧 Funções Utilitárias

def add_audit_columns(df, source_system="TEMPLATE_SYSTEM"):
    """
    Adiciona colunas de auditoria padrão para rastreabilidade
    
    Args:
        df: DataFrame PySpark
        source_system: Sistema de origem dos dados
    
    Returns:
        DataFrame com colunas de auditoria
    """
    return df.withColumn("processing_timestamp", lit(PROCESSING_TIMESTAMP)) \
             .withColumn("processing_date", lit(PROCESSING_DATE)) \
             .withColumn("source_system", lit(source_system)) \
             .withColumn("file_name", input_file_name()) \
             .withColumn("ingestion_method", lit("batch")) \
             .withColumn("record_hash", sha2(concat_ws("|", *[col(c) for c in df.columns]), 256))

def validate_data_quality(df, table_name):
    """
    Executa validações básicas de qualidade dos dados
    
    Args:
        df: DataFrame PySpark
        table_name: Nome da tabela para logs
    
    Returns:
        Dict com métricas de qualidade
    """
    total_records = df.count()
    
    # Verificar registros duplicados
    duplicate_records = df.count() - df.dropDuplicates().count()
    
    # Verificar registros com valores nulos em todas as colunas
    null_records = df.filter(
        reduce(lambda x, y: x & y, [col(c).isNull() for c in df.columns])
    ).count()
    
    quality_metrics = {
        "table_name": table_name,
        "total_records": total_records,
        "duplicate_records": duplicate_records,
        "null_records": null_records,
        "data_quality_score": (total_records - duplicate_records - null_records) / total_records * 100
    }
    
    logger.info(f"📊 Qualidade {table_name}: {quality_metrics['data_quality_score']:.2f}%")
    
    return quality_metrics

def load_source_data(file_path, file_format="csv", schema=None):
    """
    Carrega dados de arquivo usando formato especificado
    
    Args:
        file_path: Caminho do arquivo
        file_format: Formato do arquivo (csv, parquet, json)
        schema: Schema opcional para aplicar
    
    Returns:
        DataFrame PySpark
    """
    try:
        if file_format.lower() == "csv":
            reader = spark.read.option("header", "true").option("inferSchema", "true")
            if schema:
                reader = reader.schema(schema)
            df = reader.csv(file_path)
            
        elif file_format.lower() == "parquet":
            df = spark.read.parquet(file_path)
            
        elif file_format.lower() == "json":
            reader = spark.read.option("multiline", "true")
            if schema:
                reader = reader.schema(schema)
            df = reader.json(file_path)
            
        else:
            raise ValueError(f"Formato não suportado: {file_format}")
            
        logger.info(f"✅ Dados carregados: {df.count()} registros de {file_path}")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao carregar {file_path}: {str(e)}")
        raise

print("🔧 Funções utilitárias definidas")

In [ ]:
# 📊 Processamento das Tabelas Bronze

quality_report = []

for table_name, config in TABLES_CONFIG.items():
    try:
        logger.info(f"🔄 Processando tabela: {table_name}")
        
        # 1. Carregar dados de origem
        source_file_path = f"{SOURCE_PATH}{config['source_file']}"
        df_source = load_source_data(
            file_path=source_file_path,
            file_format=SOURCE_FORMAT,
            schema=config.get('schema')
        )
        
        # 2. Validar qualidade dos dados
        quality_metrics = validate_data_quality(df_source, table_name)
        quality_report.append(quality_metrics)
        
        # 3. Adicionar colunas de auditoria
        df_bronze = add_audit_columns(df_source, source_system=f"TEMPLATE_{table_name.upper()}")
        
        # 4. Definir caminho da tabela Bronze
        bronze_table_path = f"{BRONZE_PATH}/{table_name}"
        
        # 5. Salvar na camada Bronze
        # TODO: Ajustar modo de escrita conforme necessidade
        # append: adicionar novos dados
        # overwrite: substituir dados existentes
        write_mode = "append"  # ou "overwrite"
        
        df_bronze.write \
            .mode(write_mode) \
            .partitionBy(*config.get('partition_columns', [])) \
            .option("mergeSchema", "true") \
            .format("delta") \
            .save(bronze_table_path)
        
        # 6. Registrar tabela no metastore
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS bronze_{table_name}
            USING DELTA
            LOCATION '{bronze_table_path}'
        """)
        
        logger.info(f"✅ Tabela {table_name} processada com sucesso")
        
        # 7. Mostrar amostra dos dados
        print(f"\n📋 Amostra da tabela bronze_{table_name}:")
        df_bronze.limit(5).display()
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {table_name}: {str(e)}")
        # TODO: Implementar estratégia de recuperação de erro
        continue

print("\n🎉 Processamento da camada Bronze concluído!")

In [ ]:
# 📈 Relatório de Qualidade dos Dados

print("\n📊 RELATÓRIO DE QUALIDADE DOS DADOS - BRONZE LAYER")
print("=" * 60)

# Converter para DataFrame para melhor visualização
quality_df = spark.createDataFrame(
    [Row(**metrics) for metrics in quality_report]
)

# Mostrar métricas de qualidade
quality_df.select(
    "table_name",
    "total_records",
    "duplicate_records",
    "null_records",
    col("data_quality_score").cast("decimal(5,2)").alias("quality_score_pct")
).display()

# Calcular estatísticas gerais
total_records_processed = sum([m['total_records'] for m in quality_report])
average_quality_score = sum([m['data_quality_score'] for m in quality_report]) / len(quality_report)

print(f"\n📋 RESUMO GERAL:")
print(f"   📊 Total de registros processados: {total_records_processed:,}")
print(f"   🎯 Score médio de qualidade: {average_quality_score:.2f}%")
print(f"   📅 Data de processamento: {PROCESSING_DATE}")

# Identificar tabelas com problemas de qualidade
problematic_tables = [m for m in quality_report if m['data_quality_score'] < 95]
if problematic_tables:
    print(f"\n⚠️  ATENÇÃO: {len(problematic_tables)} tabela(s) com qualidade < 95%:")
    for table in problematic_tables:
        print(f"   - {table['table_name']}: {table['data_quality_score']:.2f}%")
else:
    print(f"\n✅ Todas as tabelas têm qualidade >= 95%")

In [ ]:
# 🔍 Validação Final e Testes

print("\n🔍 VALIDAÇÃO FINAL DAS TABELAS BRONZE")
print("=" * 50)

# Verificar se todas as tabelas foram criadas
bronze_tables = spark.sql("""
    SHOW TABLES LIKE 'bronze_*'
""").collect()

print(f"📋 Tabelas Bronze criadas: {len(bronze_tables)}")
for table in bronze_tables:
    table_name = table.tableName
    
    # Contar registros
    record_count = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}").collect()[0].count
    
    # Verificar partições
    try:
        partitions = spark.sql(f"SHOW PARTITIONS {table_name}").count()
    except:
        partitions = "N/A"
    
    print(f"   ✅ {table_name}: {record_count:,} registros, {partitions} partições")

# TODO: Adicionar testes específicos do seu domínio
print("\n🧪 TESTES DE INTEGRIDADE:")

# Exemplo de teste: verificar se não há datas futuras
for table_name, config in TABLES_CONFIG.items():
    table_full_name = f"bronze_{table_name}"
    try:
        future_dates = spark.sql(f"""
            SELECT COUNT(*) as count 
            FROM {table_full_name} 
            WHERE processing_date > current_date()
        """).collect()[0].count
        
        if future_dates == 0:
            print(f"   ✅ {table_name}: Sem datas futuras")
        else:
            print(f"   ⚠️  {table_name}: {future_dates} registros com datas futuras")
            
    except Exception as e:
        print(f"   ❌ {table_name}: Erro no teste - {str(e)}")

print("\n🎉 Validação da camada Bronze concluída!")
print("\n➡️  Próximo passo: Processar camada Silver com transformações e SCD Tipo 2")

## 📚 Próximos Passos

1. **Personalização**: Adapte as configurações para sua fonte de dados específica
2. **Schema**: Defina schemas explícitos para melhor performance e validação
3. **Monitoramento**: Implemente alertas para problemas de qualidade
4. **Silver Layer**: Use o template Silver para próxima camada

## 🔧 Customizações Comuns

### Para APIs REST:
```python
import requests
import json

def load_from_api(endpoint, headers=None):
    response = requests.get(endpoint, headers=headers)
    data = response.json()
    return spark.createDataFrame(data)
```

### Para Banco de Dados:
```python
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:sqlserver://server:port;database=db") \
    .option("dbtable", "schema.table") \
    .option("user", "username") \
    .option("password", "password") \
    .load()
```

### Para Streaming:
```python
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/path/to/schema") \
    .load("/path/to/streaming/data")
```